# CASMI26 Phase-4 PROBE: is spectrum→SMILES generation feasible?
**Decisive question:** for molecules that have NO structure in any candidate pool / library
(true Class-3), can a model recover the EXACT structure (InChIKey14) from MS/MS alone?

**Leak-free probe:** split by STRUCTURE (dedup inchikey14) — held-out molecules are NEVER seen
in training. Train a small spectrum→SMILES transformer; measure exact InChIKey14 match on the
held-out set.

- Held-out exact-match ≈ random → generation is not learnable with this framing → drop Phase 4,
  bank the 0.328 retrieval. (This is the field's shared null hypothesis.)
- Any measurable fraction recovered → generation is learnable → invest real GPU on it.

Fully offline on base torch. No pip, no internet.

In [ ]:
import glob, subprocess, os
_whl = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
if _whl:
    subprocess.run(['pip','install','--quiet','--no-index', _whl[0]], check=True)
else:
    print('WARN: no rdkit wheel found - exact-key eval will degrade', flush=True)
import os, sys, math, glob, time
import numpy as np
import pyarrow.parquet as pq
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

T0=time.time()
def log(m): print(f'[{time.time()-T0:6.0f}s] {m}', flush=True)

SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
MAX_MOLS=60000; VAL_FRAC=0.10
MZ_MAX=1200.0; MZ_BIN=0.05; NB=int(MZ_MAX/MZ_BIN)+1
PAD=0
D_MODEL=256; D_FF=1024; NHEADS=8; NLAYERS=4; DROPOUT=0.1; EMB_DROP=0.1
MAXLEN=85; BATCH=64; STEPS=4000; WARM=400; LR=1e-3; WDECAY=1e-5; CLIP=1.0
NBEAM=3; GEN_TOK=60; GPU='cuda' if torch.cuda.is_available() else 'cpu'
EVAL_EVERY=250; EVAL_N=120
log(f'dev={GPU} nbins={NB}')

In [ ]:
root=None
for r,_,fs in os.walk('/kaggle/input'):
    if 'train.parquet' in fs: root=r; break
assert root, 'no train.parquet'
log('input '+root)

t = pq.read_table(os.path.join(root,'train.parquet'),
                  columns=['inchikey14','normalized_smiles','precursor_mz',
                           'ms2_mzs','ms2_normalized_intensities'])
ik  = np.asarray(t.column('inchikey14').cast('string').to_pylist())
smi = np.asarray(t.column('normalized_smiles').cast('string').to_pylist())
prec= t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
mzc = t.column('ms2_mzs').combine_chunks().values.to_numpy(zero_copy_only=False).astype(np.float32)
itc = t.column('ms2_normalized_intensities').combine_chunks().values.to_numpy(zero_copy_only=False).astype(np.float32)
off = t.column('ms2_mzs').combine_chunks().offsets.to_numpy().astype(np.int64)
n=len(ik); del t
log(f'{n} spectra, {len(np.unique(ik))} unique structs')

In [ ]:
# one representative spectrum per structure
seen={}; rep=[]
for r in range(n):
    k=ik[r]
    if k and k not in seen and off[r+1]>off[r]:
        seen[k]=r; rep.append(r)
reps=np.array(rep)
rng=np.random.default_rng(SEED)
perm=rng.permutation(len(reps))
n_val=int(len(reps)*VAL_FRAC)
val_reps=reps[perm[:n_val]]; tr_reps=reps[perm[n_val:]]
if len(tr_reps)>MAX_MOLS: tr_reps=tr_reps[:MAX_MOLS]
log(f'mols total={len(reps)} train={len(tr_reps)} val={len(val_reps)}')

In [ ]:
def encode_rep(r):
    a,b=off[r],off[r+1]
    mz,it=mzc[a:b],itc[a:b]
    if len(mz)==0: return None
    keep=(mz<=prec[r]+2.0)&(mz>0)
    mz,it=mz[keep],it[keep]
    if len(mz)==0: return None
    mx=it.max()
    if mx<=0: return None
    it=np.sqrt(it/mx)
    x=np.zeros(NB,np.float32)
    idx=np.clip((mz/MZ_BIN).astype(np.int64),0,NB-1)
    np.maximum.at(x,idx,it)
    return x
chars=sorted(set(''.join(smi[reps.tolist()])))
VOCAB=['<pad>','<bos>','<eos>']+chars
c2i={c:i for i,c in enumerate(VOCAB)}
def smi_tok(s): return [c2i['<bos>']]+[c2i[ch] for ch in s]+[c2i['<eos>']]
log(f'vocab={len(VOCAB)}')

In [ ]:
class ProbeDS(Dataset):
    def __init__(self, reps_idx):
        self.Xs=[]; self.Ys=[]; self.iks=[]
        sk=0
        for r in reps_idx:
            try: x=encode_rep(r)
            except Exception: x=None
            if x is None: sk+=1; continue
            s=str(smi[r]); y=smi_tok(s)
            if len(y)>MAXLEN: continue
            self.Xs.append(x); self.Ys.append(y); self.iks.append(ik[r])
        log(f'  built {len(self.Xs)} skipped {sk}')
    def __len__(self): return len(self.Xs)
    def __getitem__(self,i): return self.Xs[i],self.Ys[i],i
trds=ProbeDS(tr_reps); vads=ProbeDS(val_reps)

In [ ]:
def pad_batch(batch):
    X=[b[0] for b in batch]; Y=[b[1] for b in batch]; I=[b[2] for b in batch]
    X=np.stack(X).astype(np.float32)
    L=max(len(y) for y in Y)
    Yp=np.zeros((len(Y),L),np.int64)
    for i,y in enumerate(Y): Yp[i,:len(y)]=y
    return torch.from_numpy(X),torch.from_numpy(Yp),I
def loader(ds,bs=BATCH,shuff=True):
    return DataLoader(ds,batch_size=bs,shuffle=shuff,collate_fn=pad_batch,
                      num_workers=0,drop_last=False)

In [ ]:
class Proj(nn.Module):
    def __init__(self,nin,d):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(nin,d),nn.GELU(),nn.Linear(d,d))
    def forward(self,x): return self.net(x)
def pos_enc(maxlen,d):
    pe=torch.zeros(maxlen,d)
    pos=torch.arange(maxlen).unsqueeze(1).float()
    i=torch.arange(d//2).unsqueeze(0).float()
    pe[:,0::2]=torch.sin(pos/10000**(2*i/d))
    pe[:,1::2]=torch.cos(pos/10000**(2*i/d))
    return pe.unsqueeze(0)
class Gen(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj=Proj(NB,D_MODEL)
        self.tok=nn.Embedding(len(VOCAB),D_MODEL)
        self.pe=nn.Parameter(pos_enc(MAXLEN,D_MODEL),requires_grad=False)
        layer=nn.TransformerDecoderLayer(D_MODEL,NHEADS,D_FF,DROPOUT,batch_first=True,norm_first=True)
        self.dec=nn.TransformerDecoder(layer,NLAYERS)
        self.emb_drop=nn.Dropout(EMB_DROP)
        self.head=nn.Linear(D_MODEL,len(VOCAB))
    def mem(self,x):
        return self.proj(x).unsqueeze(1)
    def forward(self,x,y):
        m=self.mem(x)
        t=self.tok(y[:,:-1]); L=t.size(1); t=t+self.pe[:,:L]
        t=self.emb_drop(t)
        mask=torch.triu(torch.full((L,L),float('-inf')),1).to(x.device)
        z=self.dec(t,m,tgt_mask=mask)
        return F.log_softmax(self.head(z),dim=-1)
    def generate(self,x,maxlen=GEN_TOK):
        self.eval(); B=x.size(0)
        m=self.mem(x)
        y=torch.full((B,1),c2i['<bos>'],dtype=torch.long,device=x.device)
        eos=c2i['<eos>']
        done=torch.zeros(B,dtype=torch.bool,device=x.device)
        for _ in range(maxlen):
            t=self.tok(y); L=t.size(1); t=t+self.pe[:,:L]
            mask=torch.triu(torch.full((L,L),float('-inf')),1).to(x.device)
            z=self.dec(t,m,tgt_mask=mask)
            nxt=self.head(z[:,-1]).argmax(-1)
            y=torch.cat([y,nxt.unsqueeze(1)],1)
            done |= (nxt==eos)
            if done.all(): break
        self.train()
        return y
model=Gen().to(GPU)
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WDECAY)
sch=torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s+1)/WARM,1.0))
log(f'params={sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
def dec_smiles(t):
    out=[]
    for tok in t[1:]:
        if tok==c2i['<eos>']: break
        out.append(VOCAB[tok])
    return ''.join(out)
def key14(s):
    try:
        m=Chem.MolFromSmiles(s)
        return None if m is None else Chem.MolToInchiKey(m)[:14]
    except Exception:
        return None
def eval_probe():
    model.eval()
    n=min(EVAL_N,len(vads))
    xb=np.stack([vads.Xs[i] for i in range(n)]).astype(np.float32)
    xb=torch.from_numpy(xb).to(GPU)
    with torch.no_grad():
        g=model.generate(xb,maxlen=GEN_TOK).cpu().tolist()
    exact=valid=0
    for i,tk in enumerate(g):
        pred=dec_smiles(tk); k=key14(pred)
        if k is not None:
            valid+=1
            if k==vads.iks[i]: exact+=1
    model.train()
    return exact,n,valid
tr=loader(trds); va=loader(vads,shuff=False)
it=iter(tr)
for step in range(STEPS):
    model.train()
    try: x,y,_=next(it)
    except StopIteration:
        it=iter(tr); x,y,_=next(it)
    x=x.to(GPU); y=y.to(GPU)
    out=model(x,y)
    loss=F.nll_loss(out.reshape(-1,len(VOCAB)), y[:,1:].reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(),CLIP)
    opt.step(); sch.step()
    if step%EVAL_EVERY==0:
        e,n,v=eval_probe()
        log(f'step {step} loss={loss.item():.3f} VAL exact={e}/{n} valid={v} ({100*e/max(1,n):.1f}%)')

## Verdict
Held-out exact InChIKey14 match on structurally-unseen molecules. If this stays ~0 while training
loss falls, the model memorizes train SMILES but cannot generalize → spectrum→SMILES generation
is not feasible at this size. A measurable %, and the probe is worth scaling.